# Generate 100 AI-Generated Car Exterior Images (3 model checkpoints)

**FOML project — custom car-image dataset**

This notebook generates **100 separate images** of ordinary car exteriors using three different model checkpoints:

| Model key | Hugging Face model | Count |
|---|---|---:|
| `sd15` | `stable-diffusion-v1-5/stable-diffusion-v1-5` | 34 |
| `sdxl_turbo` | `stabilityai/sdxl-turbo` | 33 |
| `tiny_sd` | `segmind/tiny-sd` | 33 |

**Fix included:** the previous PixArt-Sigma repository returned a 404 for `model_index.json` in the runtime. The third checkpoint has been replaced with Segmind Tiny-SD, whose model card documents Diffusers usage. The notebook can migrate the existing generation plan and resume the 33 remaining images, provided your current Colab runtime and its `/content/car_ai_generated_multimodel` folder still exist. Tiny-SD is another checkpoint, not a completely independent commercial image-generation service.

- Run in Google Colab with a GPU selected: **Runtime → Change runtime type → T4 GPU**, if available.
- The notebook records model ID, prompt, seed and image path in a CSV.
- Car types, viewpoints, environments and lighting are varied.
- Review every image manually before training. Remove malformed or off-topic images from the *approved training list*, and document the review.
- Check each model's license/terms before redistributing generated outputs.
- The notebook saves images one by one and can resume when the Colab runtime storage has not been reset.

Model cards: [Stable Diffusion v1.5](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5), [SDXL Turbo](https://huggingface.co/stabilityai/sdxl-turbo), [Segmind Tiny-SD](https://huggingface.co/segmind/tiny-sd).


## 1. Install the image-generation libraries

Run this cell first. It avoids the `torchao` import conflict encountered in the earlier notebook. It does not reinstall PyTorch.

In [ ]:
!pip uninstall -y torchao >/dev/null 2>&1 || true
!pip install -q "diffusers>=0.36,<0.41" "transformers>=4.45,<5" "accelerate>=1.0,<2" "safetensors>=0.4" "sentencepiece>=0.2" "Pillow>=10.0"

## 2. Check the runtime

A GPU is strongly recommended for this notebook.

In [ ]:
import torch, sys, os
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU is active. In Colab, choose Runtime > Change runtime type > T4 GPU, "
        "then run this cell again. Image generation on CPU is likely to be too slow."
    )

## 3. Configure the output folder and model balance

In [ ]:
from pathlib import Path
import gc, csv, json, random, time, traceback
from datetime import datetime, timezone
from PIL import Image, ImageOps, ImageDraw
import pandas as pd
from IPython.display import display

# All generated images will be saved here in the Colab runtime.
OUTPUT_DIR = Path("/content/car_ai_generated_multimodel")
IMAGE_DIR = OUTPUT_DIR / "ai"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PLAN_PATH = OUTPUT_DIR / "generation_plan.csv"
METADATA_PATH = OUTPUT_DIR / "metadata.csv"
ERRORS_PATH = OUTPUT_DIR / "generation_errors.csv"

IMAGE_SIZE = 512
MASTER_SEED = 20261028

MODEL_CONFIG = {
    "sd15": {
        "display_name": "Stable Diffusion v1.5",
        "model_id": "stable-diffusion-v1-5/stable-diffusion-v1-5",
        "count": 34,
    },
    "sdxl_turbo": {
        "display_name": "SDXL Turbo",
        "model_id": "stabilityai/sdxl-turbo",
        "count": 33,
    },
    "tiny_sd": {
        "display_name": "Segmind Tiny-SD",
        "model_id": "segmind/tiny-sd",
        "count": 33,
    },
}

print("Images requested:", sum(v["count"] for v in MODEL_CONFIG.values()))
print("Output folder:", OUTPUT_DIR)


## 4. Make a balanced set of 100 car prompts

The plan combines 10 car descriptions with 10 viewpoints/settings. That gives 100 unique prompt combinations. The combination list and model assignments are shuffled with fixed seeds and written to `generation_plan.csv` so reruns can continue the same plan.

In [ ]:
car_descriptions = [
    "a silver compact hatchback",
    "a white midsize sedan",
    "a dark blue electric sedan",
    "a red two-door sports coupe",
    "a black midsize SUV",
    "a grey pickup truck",
    "a green vintage coupe",
    "a beige station wagon",
    "an orange performance hatchback",
    "a white family minivan",
]

scene_descriptions = [
    "front three-quarter view, parked on a quiet residential street in soft morning daylight",
    "clean side profile, parked in an ordinary outdoor parking lot under bright overcast skies",
    "rear three-quarter view, parked beside a suburban house driveway in natural daylight",
    "straight front view, parked on a tree-lined urban street in cloudy afternoon light",
    "side view, parked at the edge of a public parking area after light rain, realistic wet pavement",
    "front three-quarter view, parked outside a plain modern building, diffuse daylight, uncluttered background",
    "rear view, parked in a quiet residential lane with natural shadows and realistic surroundings",
    "side profile, parked near a low wall and trees in late afternoon sunlight",
    "front three-quarter view, parked in an ordinary dealership lot with no visible signs or banners",
    "rear three-quarter view, parked along a suburban curb on a clear but slightly hazy day",
]

# Kept shorter than the CLIP 77-token limit used by Stable Diffusion models.
# This avoids truncating the important portion of the prompt.
base_prompt = (
    "Photorealistic everyday photograph of {car}, {scene}. Entire car visible, "
    "natural camera perspective, believable proportions, realistic tires, windows, "
    "reflections and shadows. No people, text, watermark, collage, illustration, "
    "cartoon or obvious 3D-render look."
)

if PLAN_PATH.exists():
    plan = pd.read_csv(PLAN_PATH)
    print("Loaded existing plan to support resuming:", len(plan), "rows")

    # Migrate the unfinished PixArt-Sigma rows from the previous notebook. That
    # repository returned a 404 for model_index.json, so use Tiny-SD for those
    # remaining prompt IDs instead. Previously generated SD1.5/SDXL files stay put.
    if "model_key" in plan.columns and plan["model_key"].astype(str).eq("pixart_sigma").any():
        mask = plan["model_key"].astype(str).eq("pixart_sigma")
        if METADATA_PATH.exists():
            old_meta = pd.read_csv(METADATA_PATH)
            if "model_key" in old_meta.columns and old_meta["model_key"].astype(str).eq("pixart_sigma").any():
                raise RuntimeError(
                    "The metadata already contains PixArt-Sigma outputs. Do not relabel them as Tiny-SD. "
                    "Please share the metadata.csv so we can migrate safely."
                )
        plan.loc[mask, "model_key"] = "tiny_sd"
        plan.loc[mask, "model_name"] = MODEL_CONFIG["tiny_sd"]["display_name"]
        plan.loc[mask, "model_id"] = MODEL_CONFIG["tiny_sd"]["model_id"]
        plan.loc[mask, "relative_path"] = plan.loc[mask, "image_id"].map(lambda x: f"ai/tiny_sd/{x}.png")
        plan.loc[mask, "prompt"] = plan.loc[mask].apply(
            lambda r: base_prompt.format(car=r["car_description"], scene=r["scene_description"]), axis=1
        )
        plan.to_csv(PLAN_PATH, index=False)
        print("Migrated the 33 unfinished PixArt-Sigma plan rows to Segmind Tiny-SD.")
else:
    combinations = [(car, scene) for car in car_descriptions for scene in scene_descriptions]
    random.Random(MASTER_SEED).shuffle(combinations)

    model_assignments = []
    for model_key, config in MODEL_CONFIG.items():
        model_assignments.extend([model_key] * config["count"])
    random.Random(MASTER_SEED + 1).shuffle(model_assignments)

    rows = []
    for i, ((car, scene), model_key) in enumerate(zip(combinations, model_assignments), start=1):
        prompt = base_prompt.format(car=car, scene=scene)
        rows.append({
            "image_id": f"CAR_AI_{i:03d}",
            "car_description": car,
            "scene_description": scene,
            "model_key": model_key,
            "model_name": MODEL_CONFIG[model_key]["display_name"],
            "model_id": MODEL_CONFIG[model_key]["model_id"],
            "prompt": prompt,
            "seed": MASTER_SEED + i,
            "relative_path": f"ai/{model_key}/CAR_AI_{i:03d}.png",
        })
    plan = pd.DataFrame(rows)
    plan.to_csv(PLAN_PATH, index=False)

print("Prompt-plan rows:", len(plan))
display(plan.groupby(["model_key", "model_name"]).size().rename("images_planned").reset_index())
display(plan[["image_id", "model_name", "car_description", "scene_description"]].head(8))


## 5. Load and run one model at a time

Only one model is kept in memory at a time. The previous run successfully created the 34 Stable Diffusion v1.5 and 33 SDXL Turbo images, then failed while loading the PixArt-Sigma repository because its expected `model_index.json` URL returned 404. The code above migrates the 33 unfinished plan rows to Segmind Tiny-SD. Run the generation cell again; existing files are skipped, and only missing files should be generated.


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, AutoPipelineForText2Image

DEVICE = "cuda"
DTYPE = torch.float16
NEGATIVE_PROMPT = (
    "cartoon, illustration, 3d render, toy, miniature, distorted car, "
    "malformed wheels, extra wheels, duplicated car, text, watermark, people, cropped car"
)

def load_pipeline(model_key):
    """Load the requested model and return its Diffusers pipeline."""
    model_id = MODEL_CONFIG[model_key]["model_id"]
    print(f"\nLoading {MODEL_CONFIG[model_key]['display_name']} ({model_id}) ...")

    if model_key == "sd15":
        pipe = StableDiffusionPipeline.from_pretrained(
            model_id, torch_dtype=DTYPE, use_safetensors=True
        )
        pipe = pipe.to(DEVICE)
        if hasattr(pipe, "enable_attention_slicing"):
            pipe.enable_attention_slicing()

    elif model_key == "sdxl_turbo":
        pipe = AutoPipelineForText2Image.from_pretrained(
            model_id, torch_dtype=DTYPE, variant="fp16", use_safetensors=True
        )
        pipe = pipe.to(DEVICE)

    elif model_key == "tiny_sd":
        # Tiny-SD is a compact Stable-Diffusion-style checkpoint based on
        # Realistic Vision V4. It is a separate checkpoint from SD1.5/SDXL Turbo.
        # This repository currently provides PyTorch .bin weights rather than safetensors,
        # so do not force use_safetensors=True for this trusted checkpoint.
        pipe = AutoPipelineForText2Image.from_pretrained(
            model_id, torch_dtype=DTYPE
        )
        pipe = pipe.to(DEVICE)
        if hasattr(pipe, "enable_attention_slicing"):
            pipe.enable_attention_slicing()

    else:
        raise ValueError(f"Unknown model key: {model_key}")

    return pipe


def generate_one(pipe, model_key, prompt, seed):
    """Generate one 512x512 image using the selected model's settings."""
    generator = torch.Generator(device="cuda").manual_seed(int(seed))

    if model_key == "sd15":
        result = pipe(
            prompt=prompt, negative_prompt=NEGATIVE_PROMPT,
            width=IMAGE_SIZE, height=IMAGE_SIZE,
            num_inference_steps=25, guidance_scale=7.0, generator=generator,
        )
    elif model_key == "sdxl_turbo":
        # SDXL Turbo is designed for a small number of steps and guidance_scale=0.
        result = pipe(
            prompt=prompt, width=IMAGE_SIZE, height=IMAGE_SIZE,
            num_inference_steps=4, guidance_scale=0.0, generator=generator,
        )
    elif model_key == "tiny_sd":
        result = pipe(
            prompt=prompt, negative_prompt=NEGATIVE_PROMPT,
            width=IMAGE_SIZE, height=IMAGE_SIZE,
            num_inference_steps=25, guidance_scale=7.0, generator=generator,
        )
    else:
        raise ValueError(f"Unknown model key: {model_key}")

    image = result.images[0].convert("RGB")
    if image.size != (IMAGE_SIZE, IMAGE_SIZE):
        image = ImageOps.fit(image, (IMAGE_SIZE, IMAGE_SIZE), method=Image.Resampling.LANCZOS)
    return image

print("Model-loading functions are ready.")


## 6. Generate the 100 images

This cell saves every successful image immediately and updates `metadata.csv` after each image. If the session is interrupted but the runtime storage survives, rerun this cell to skip files already generated.

In [ ]:
metadata_columns = [
    "image_id", "label", "model_key", "model_name", "model_id", "prompt", "seed",
    "relative_path", "width", "height", "file_format", "generated_at_utc", "review_status"
]

if METADATA_PATH.exists():
    metadata = pd.read_csv(METADATA_PATH)
else:
    metadata = pd.DataFrame(columns=metadata_columns)

recorded_ids = set(metadata.get("image_id", pd.Series(dtype=str)).astype(str))

# Process the models one-by-one so we do not keep multiple generators in GPU memory.
for model_key in MODEL_CONFIG:
    model_rows = plan[plan["model_key"] == model_key].to_dict("records")
    unfinished = []
    for row in model_rows:
        out_path = OUTPUT_DIR / row["relative_path"]
        if out_path.exists():
            # Recover the metadata record if an earlier run saved the image but stopped before logging it.
            if row["image_id"] not in recorded_ids:
                with Image.open(out_path) as check_img:
                    width, height = check_img.size
                record = {
                    **{k: row[k] for k in ["image_id", "model_key", "model_name", "model_id", "prompt", "seed", "relative_path"]},
                    "label": "ai_generated", "width": width, "height": height,
                    "file_format": "PNG", "generated_at_utc": "recovered_from_existing_file",
                    "review_status": "needs_visual_review",
                }
                metadata = pd.concat([metadata, pd.DataFrame([record])], ignore_index=True)
                metadata.to_csv(METADATA_PATH, index=False)
                recorded_ids.add(row["image_id"])
            continue
        unfinished.append(row)

    if not unfinished:
        print(f"\n{model_key}: all {len(model_rows)} images already exist; skipping model load.")
        continue

    pipe = load_pipeline(model_key)
    print(f"Generating {len(unfinished)} images with {MODEL_CONFIG[model_key]['display_name']}...")
    consecutive_errors = 0

    for idx, row in enumerate(unfinished, start=1):
        out_path = OUTPUT_DIR / row["relative_path"]
        out_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            image = generate_one(pipe, model_key, row["prompt"], row["seed"])
            image.save(out_path, format="PNG", optimize=True)
            record = {
                **{k: row[k] for k in ["image_id", "model_key", "model_name", "model_id", "prompt", "seed", "relative_path"]},
                "label": "ai_generated",
                "width": image.width,
                "height": image.height,
                "file_format": "PNG",
                "generated_at_utc": datetime.now(timezone.utc).isoformat(),
                "review_status": "needs_visual_review",
            }
            metadata = pd.concat([metadata, pd.DataFrame([record])], ignore_index=True)
            metadata.to_csv(METADATA_PATH, index=False)
            recorded_ids.add(row["image_id"])
            consecutive_errors = 0
            print(f"[{idx}/{len(unfinished)}] Saved {row['image_id']} using {model_key}")
        except Exception as exc:
            consecutive_errors += 1
            error_record = {
                "image_id": row["image_id"], "model_key": model_key,
                "model_id": row["model_id"], "error": repr(exc),
                "logged_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            error_df = pd.DataFrame([error_record])
            if ERRORS_PATH.exists():
                error_df.to_csv(ERRORS_PATH, mode="a", header=False, index=False)
            else:
                error_df.to_csv(ERRORS_PATH, index=False)
            print(f"ERROR for {row['image_id']}: {exc}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            # Repeated errors usually indicate an installation/memory problem; stop this model cleanly.
            if consecutive_errors >= 3:
                print("Three consecutive failures. Stopping this model. Fix the error and rerun to resume.")
                break

    del pipe
    gc.collect()
    torch.cuda.empty_cache()

# Summary
if METADATA_PATH.exists():
    metadata = pd.read_csv(METADATA_PATH)
    successful_paths = [OUTPUT_DIR / p for p in metadata["relative_path"].astype(str) if (OUTPUT_DIR / p).exists()]
    print("\nSuccessful image files:", len(successful_paths), "/ 100")
    display(metadata.groupby(["model_key", "model_name"]).size().rename("saved_images").reset_index())
    if len(successful_paths) < 100:
        print("Some images are still missing. Correct any errors above and rerun this cell; existing files will be kept.")
else:
    print("No images were generated yet. Check the errors above.")

## 7. Review the outputs visually

Check the thumbnails and the individual images before training. Generators can produce malformed wheels, partial cars, strange reflections or unwanted text. Review images at full size. Mark approved files in the metadata only after checking; do not silently delete files without recording the decision.


In [ ]:
from PIL import Image, ImageOps, ImageDraw
import math

metadata = pd.read_csv(METADATA_PATH) if METADATA_PATH.exists() else pd.DataFrame()
valid_rows = []
if not metadata.empty:
    for _, row in metadata.iterrows():
        p = OUTPUT_DIR / str(row["relative_path"])
        if p.exists():
            valid_rows.append((row, p))

cols = 10
thumb_size = 120
label_height = 28
rows_n = math.ceil(len(valid_rows) / cols) if valid_rows else 1
sheet = Image.new("RGB", (cols * thumb_size, rows_n * (thumb_size + label_height)), "white")
draw = ImageDraw.Draw(sheet)

for i, (row, path) in enumerate(valid_rows):
    with Image.open(path) as im:
        thumb = ImageOps.fit(im.convert("RGB"), (thumb_size, thumb_size), method=Image.Resampling.LANCZOS)
    x = (i % cols) * thumb_size
    y = (i // cols) * (thumb_size + label_height)
    sheet.paste(thumb, (x, y))
    draw.text((x + 3, y + thumb_size + 4), f"{row['image_id']} | {row['model_key']}", fill="black")

contact_sheet_path = OUTPUT_DIR / "contact_sheet.jpg"
sheet.save(contact_sheet_path, quality=90)
print("Contact sheet:", contact_sheet_path)
display(sheet.resize((1000, max(250, int(sheet.height * 1000 / sheet.width)))))

## 8. Create the final ZIP for the car project

The ZIP contains the individual images, `metadata.csv`, the generation plan, contact sheet and a short README. The metadata marks each image `needs_visual_review`; mark it `approved` only after visual inspection. If fewer than 100 valid images remain, correct the issue and rerun before using this as the final AI class.


In [ ]:
readme_text = """AI-GENERATED CAR EXTERIOR IMAGE COLLECTION

Purpose: custom educational dataset for a college FOML project.
Contents: generated car exterior images from three different model checkpoints.
Label: ai_generated.
Checkpoints: Stable Diffusion v1.5, SDXL Turbo, Segmind Tiny-SD.

IMPORTANT:
1. Review every image for subject relevance and obvious generation defects before training.
2. metadata.csv records the image source/model, prompt, seed, and review status.
3. Review the model licenses and terms before redistributing outputs or using them beyond education.
4. This collection alone is not the complete classifier dataset. Collect a comparable set of real car exterior photos and preprocess both classes identically.
5. Do not include this data in a public GitHub repository unless you have reviewed image/model sharing terms and file-size implications.
6. Tiny-SD is a separate checkpoint, but not a separate commercial generation service or a fully independent architecture from Stable Diffusion-style models.
"""
(OUTPUT_DIR / "README.txt").write_text(readme_text, encoding="utf-8")

import shutil
zip_base = "/content/car_ai_generated_multimodel"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=OUTPUT_DIR)
print("Created:", zip_path)
print("Metadata rows:", len(metadata) if not metadata.empty else 0)
if not metadata.empty:
    print("Files on disk:", sum((OUTPUT_DIR / p).exists() for p in metadata["relative_path"].astype(str)))

from google.colab import files
files.download(zip_path)


## 9. After generation: what to do next

1. Inspect the contact sheet and review all images at full size.
2. Ensure the generated counts are approximately **34 / 33 / 33** and every image's source is recorded.
3. Collect the real car photographs separately; vary cars, angles, backgrounds and lighting.
4. Apply the same final preprocessing to real and AI images (for example, 256×256 JPEGs) after preserving originals.
5. Keep photos of the same real car or related series in the same split. For generalization testing, reserve a generator not used for training.

These are three **model checkpoints**, not three commercial image-generation websites. This improves checkpoint diversity compared with using one checkpoint for every image, but does not guarantee generalization to tools such as Midjourney or ChatGPT.
